# Fake News Detector - Colab Training

This notebook trains all model×embedding×dataset combinations on Google Colab, persisting every artifact to Google Drive so nothing is lost on session timeout.

## Workflow

1. Configuration
2. Mount Google Drive
3. Clone or update repository
4. Install dependencies
5. Create Drive folders
6. Upload raw datasets and embeddings
7. Validate and preprocess datasets
8. Configure project symlinks (Drive ↔ project)
9. Train models
10. Collect web data
11. Clean web data
12. Evaluate on web data
13. Analyze results

## 1 - Configuration

In [18]:
from pathlib import Path

GITHUB_USERNAME = "wgrzesik"
REPO_NAME = "fake-news-detector"
BRANCH = "develop"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

DRIVE_ROOT = Path("/content/drive/MyDrive/fake-news-results")
PROJECT_DIR = Path(f"/content/{REPO_NAME}")

print("Bootstrap configuration ready")
print(f"Repository: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Project dir: {PROJECT_DIR}")

Bootstrap configuration ready
Repository: https://github.com/wgrzesik/fake-news-detector.git
Branch: develop
Drive root: /content/drive/MyDrive/fake-news-results
Project dir: /content/fake-news-detector


## 2 - Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3 - Clone or update repository
First run clones; subsequent runs pull the latest changes. After pushing code changes locally, just re-run this cell.

In [22]:
import os

if PROJECT_DIR.is_dir():
    print("Repo exists - pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git log --oneline -3

Repo exists - pulling latest changes...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 9 (delta 7), reused 9 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 929 bytes | 103.00 KiB/s, done.
From https://github.com/wgrzesik/fake-news-detector
 * branch            develop    -> FETCH_HEAD
   d288388..575547a  develop    -> origin/develop
Updating d288388..575547a
Fast-forward
 research/analyze_results.py    | 14 +++++++-------
 research/colab/colab_setup.py  |  2 +-
 research/train_models.py       |  3 +--
 research/web/clean_web_data.py | 29 -----------------------------
 4 files changed, 9 insertions(+), 39 deletions(-)
/content/fake-news-detector
575547a (HEAD -> develop, origin/develop, origin/HEAD) small refactors
d288388 update architecture.md
6af58ed update README.md


## 4 - Install dependencies

In [4]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 67.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.6 MB/s eta 0:00:00
  

## 5 - Create Drive folders
Create the required Google Drive folder structure for raw data, processed data, embeddings, and web data.

In [5]:
RAW_REQUIREMENTS = {
    "ISOT": ["True.csv", "Fake.csv"],
    "LIAR": ["train.tsv", "test.tsv", "valid.tsv"],
    "WELFake": ["data.csv"],
}

REQUIRED_EMBEDDINGS = [
    DRIVE_ROOT / "datasets" / "embeddings" / "glove.6B.100d.txt",
]

required_dirs = [
    DRIVE_ROOT / "datasets" / "raw" / "ISOT",
    DRIVE_ROOT / "datasets" / "raw" / "LIAR",
    DRIVE_ROOT / "datasets" / "raw" / "WELFake",
    DRIVE_ROOT / "datasets" / "processed" / "ISOT",
    DRIVE_ROOT / "datasets" / "processed" / "LIAR",
    DRIVE_ROOT / "datasets" / "processed" / "WELFake",
    DRIVE_ROOT / "datasets" / "embeddings",
    DRIVE_ROOT / "datasets" / "web_scraped_data" / "processed",
]

for directory in required_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print("Drive folder structure created.")

Drive folder structure created.


## 6 - Upload raw datasets and embeddings
Upload files to Google Drive before preprocessing:

Raw datasets:
- ISOT: `/content/drive/MyDrive/fake-news-results/datasets/raw/ISOT/`:
`True.csv`, `Fake.csv`
- LIAR: `/content/drive/MyDrive/fake-news-results/datasets/raw/LIAR/`: `train.tsv`, `test.tsv`, `valid.tsv`
- WELFake: `/content/drive/MyDrive/fake-news-results/datasets/raw/WELFake/`: `data.csv`

Embedding file:
- GloVe: `/content/drive/MyDrive/fake-news-results/datasets/embeddings/glove.6B.100d.txt`

## 7 - Validate and preprocess datasets

In [17]:
import os
import shutil
import subprocess
import sys

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

missing_raw = []
for dataset, filenames in RAW_REQUIREMENTS.items():
    raw_dir = DRIVE_ROOT / "datasets" / "raw" / dataset
    for filename in filenames:
        file_path = raw_dir / filename
        if not file_path.exists():
            missing_raw.append(file_path)

missing_embeddings = [path for path in REQUIRED_EMBEDDINGS if not path.exists()]

if missing_raw or missing_embeddings:
    if missing_raw:
        print("Missing required raw dataset files:")
        for path in missing_raw:
            print(f" - {path}")
    if missing_embeddings:
        print("\nMissing required embedding files:")
        for path in missing_embeddings:
            print(f" - {path}")
    raise FileNotFoundError("Upload missing raw datasets/embeddings to Drive and rerun this cell.")

print("All required raw datasets and embeddings found.")

for dataset in RAW_REQUIREMENTS:
    source = DRIVE_ROOT / "datasets" / "raw" / dataset
    target = PROJECT_DIR / "research" / dataset

    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)

    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)
    print(f"{target} -> {source}")

run([sys.executable, "-m", "research.preprocess_datasets"], cwd=PROJECT_DIR)

for dataset in RAW_REQUIREMENTS:
    processed_dir = DRIVE_ROOT / "datasets" / "processed" / dataset
    processed_dir.mkdir(parents=True, exist_ok=True)

    for split in ["train.csv", "test.csv", "val.csv"]:
        source_file = PROJECT_DIR / "research" / dataset / split
        if not source_file.exists():
            raise FileNotFoundError(f"Expected preprocessing output not found: {source_file}")
        destination = processed_dir / split
        shutil.copy2(source_file, destination)
        print(f"Copied: {source_file} -> {destination}")

print("Preprocessing stage complete.")

All required raw datasets and embeddings found.
/content/fake-news-detector/research/ISOT -> /content/drive/MyDrive/fake-news-results-4/datasets/raw/ISOT
/content/fake-news-detector/research/LIAR -> /content/drive/MyDrive/fake-news-results-4/datasets/raw/LIAR
/content/fake-news-detector/research/WELFake -> /content/drive/MyDrive/fake-news-results-4/datasets/raw/WELFake
$ /usr/bin/python3 -m research.preprocess_datasets
Copied: /content/fake-news-detector/research/ISOT/train.csv -> /content/drive/MyDrive/fake-news-results-4/datasets/processed/ISOT/train.csv
Copied: /content/fake-news-detector/research/ISOT/test.csv -> /content/drive/MyDrive/fake-news-results-4/datasets/processed/ISOT/test.csv
Copied: /content/fake-news-detector/research/ISOT/val.csv -> /content/drive/MyDrive/fake-news-results-4/datasets/processed/ISOT/val.csv
Copied: /content/fake-news-detector/research/LIAR/train.csv -> /content/drive/MyDrive/fake-news-results-4/datasets/processed/LIAR/train.csv
Copied: /content/fake-n

## 8 - Configure project symlinks (Drive ↔ project)
Links datasets/embeddings into the project and output dirs out to Drive.

In [23]:
!python -m research.colab.colab_setup --project {PROJECT_DIR} --drive {DRIVE_ROOT}


Colab Setup
Project: /content/fake-news-detector
Drive: /content/drive/MyDrive/fake-news-results

[1/5] Linking OUTPUT directories (project -> Drive) …
saved_models -> /content/drive/MyDrive/fake-news-results/saved_models
experiments -> /content/drive/MyDrive/fake-news-results/experiments
outputs -> /content/drive/MyDrive/fake-news-results/outputs
research/configs/web_scraped_data -> /content/drive/MyDrive/fake-news-results/research/configs/web_scraped_data

[2/5] Linking OUTPUT files (project -> Drive) …
mlflow.db -> /content/drive/MyDrive/fake-news-results/mlflow.db
optuna.db -> /content/drive/MyDrive/fake-news-results/optuna.db

[3/5] Linking INPUT data (Drive -> project) …
SKIP  datasets/processed  (not found on Drive - upload it first)
SKIP  datasets/embeddings  (not found on Drive - upload it first)
SKIP  datasets/web_scraped_data  (not found on Drive - upload it first)

[4/5] Discovering web-scraped data …
No web-scraped CSV files found at: /content/drive/MyDrive/fake-news-resu

## 9 - Train models
Each cell trains one dataset. Re-runnable: Optuna studies use load_if_exists=True, so interrupted runs resume where they left off.

Key Hydra overrides you can add:

- `datasets_list=[ISOT,LIAR]` — subset of datasets
- `models_to_optimize=[svm,xgb]` — subset of models
- `embeddings_to_use=[tfidf,glove]` — subset of embeddings
- `optuna.n_trials=20` — more tuning trials
- `hydra.job.chdir=Falsec — disables dynamic output folders to keep your Drive symlinks and relative paths intact

example:
```
  !python -m research.train_models \
  datasets_list=[LIAR] \
  models_to_optimize=[bert] \
  embeddings_to_use=[bert-base-uncased] \
  hydra.job.chdir=False

In [8]:
  !python -m research.train_models \
  datasets_list=[LIAR] \
  models_to_optimize=[lr] \
  embeddings_to_use=[glove] \
  hydra.job.chdir=False


LARGE-SCALE EXPERIMENT RUNNER
Datasets: 1 -> ['LIAR']
Models: 1 -> ['lr']
Embeddings: 1 -> ['glove']
Preprocessing: classic
Total combinations: 1
Optuna trials per model: 10
MLflow experiment: fake_news_detection
Results directory: ./experiments

2026/05/24 16:03:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/24 16:03:43 INFO mlflow.store.db.utils: Updating database tables
2026/05/24 16:03:48 INFO mlflow.tracking.fluent: Experiment with name 'fake_news_detection' does not exist. Creating a new experiment.

####################################################################################################
[1/1] DATASET: LIAR
####################################################################################################

[Loading Data]
Train samples: 10240
Test samples: 1267

[Preprocessing] CLASSIC (for model 'lr')
Preprocessing done (0.47s)

[1/1] LIAR | LR | GLOVE
--------------------------------------------------------------------------------

## 10 - Collect web data
Scrapes real-world news articles, preprocesses with the same pipeline as training data.
Outputs three CSVs to `experiments/web_test_results/` (title, excerpt, full text) for
generalization testing on out-of-distribution data.

In [11]:
!python -m research.web.collect_web


[SCRAPING MODE]
Real per source: 96
Fake per source: 416

[Collecting Real News] 52 sources
Fetched 43 from BBC_World (full text: 43)
Fetched 21 from BBC_Tech (full text: 21)
Fetched 60 from NYT_World (full text: 21)
Fetched 20 from NYT_US (full text: 0)
Fetched 10 from NPR (full text: 10)
Fetched 45 from Guardian_World (full text: 45)
Fetched 33 from Guardian_US (full text: 33)
Fetched 25 from AlJazeera (full text: 25)
Fetched 3 from WashingtonPost (full text: 0)
Fetched 20 from WallStreetJournal (full text: 0)
Fetched 25 from ABC_News (full text: 24)
Fetched 30 from CBS_News (full text: 28)
Fetched 10 from SkyNews (full text: 0)
Fetched 17 from NYTimes (full text: 0)
Fetched 25 from FoxNews (full text: 25)
Fetched 25 from NBCNews (full text: 25)
Fetched 96 from DailyMail (full text: 96)
Fetched 96 from Independent (full text: 96)
Fetched 23 from NyPost (full text: 23)
Fetched 10 from Express (full text: 10)
Fetched 25 from FinancialTimes (full text: 0)
Fetched 10 from NationalPublic

# 11 - Clean web data
This script cleans web-scraped news data and prepares it for later model evaluation. It removes missing values, Unicode artifacts, duplicates, very short articles, and optionally non-English texts. The cleaned data is then split into separate CSV files for titles, full article texts, and short texts, each containing only the `text` and `label` columns.

In [13]:
!python -m research.web.clean_web_data


Loading data from: research/configs/web_scraped_data/web_scraped_news.csv

Input Dataset Statistics
----------------------------------------
Total samples: 1745
Real news (1): 1586
Fake news (0): 159
Avg text length: 3561 chars
Min text length: 3 chars
Max text length: 53613 chars
Unique sources: 62

CLEANING WEB-SCRAPED DATA
Initial samples: 1745
  Removed 190 rows with missing text/label

[Step 1/4] Cleaning Unicode artifacts...
Cleaning text: 100% 1555/1555 [00:00<00:00, 2211.68it/s]
/content/fake-news-detector/research/web/clean_web_data.py:205: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].progress_apply(remove_unicode_artifacts)
/content/fake-news-detector/research/web/clean_web_data.py:209: SettingWithCopyWarning: 

## 12 - Evaluate on web data
Runs all trained models against collected web data. Compares across datasets (ISOT, LIAR, WELFake),
model types (SVM, XGBoost, NN), embeddings (TF-IDF, GloVe, BERT), and text types (title, excerpt, full text).
Outputs predictions to `experiments/web_test_results/results_*.csv`.

In [14]:
!python -m research.evaluate_web


WEB DATA TESTING

Configuration loaded:
  Datasets: ['ISOT', 'LIAR', 'WELFake']
  Models: ['dt', 'knn', 'lr', 'mnb', 'nb', 'rf', 'svm', 'xgb', 'lstm', 'gru', 'bilstm', 'cnn', 'bert', 'roberta', 'fakebert', 'distilbert']
  Embeddings: ['tfidf', 'bow', 'glove', 'word2vec', 'bert-base-uncased', 'roberta-base', 'distilbert-base-uncased']
  Preprocessing: classic
  Results dir: ./experiments/web_test_results
  Processed dir: research/configs/web_scraped_data/processed
  Text types: ['title', 'text', 'short_text']


####################################################################################################
  EVALUATING TEXT TYPE: TITLE
  Source: research/configs/web_scraped_data/processed/web_title.csv
####################################################################################################
Loaded 1432 valid samples for testing (text_type='title').


TESTING DATASET: ISOT | text_type: title
Dataset directory not found: /content/drive/MyDrive/fake-news-results-4/saved_mod

## 13 - Analyze results
Aggregates all results into reports and visualizations. Computes F1/Precision/Recall across
all combinations, calculates generalization gaps, and generates charts saved to `presentation_charts/`.
Check `GENERALIZATION_SUMMARY.md` for a human-readable summary of top-performing models.

In [15]:
!python -m research.analyze_results

[1/5] Loading training data...
  Training results: 1 models

[2/5] Loading web results for text types: ['title', 'text', 'short_text']...
  title       : 1 models
  text        : 1 models
  short_text  : 1 models

[3/5] Analysis for text_type='title'...

  -- LIAR / title --
    OK  Results saved to: /content/fake-news-detector/experiments/results/title/liar
  OK  Global results saved to: /content/fake-news-detector/experiments/results/title/all

[3/5] Analysis for text_type='text'...

  -- LIAR / text --
    OK  Results saved to: /content/fake-news-detector/experiments/results/text/liar
  OK  Global results saved to: /content/fake-news-detector/experiments/results/text/all

[3/5] Analysis for text_type='short_text'...

  -- LIAR / short_text --
    OK  Results saved to: /content/fake-news-detector/experiments/results/short_text/liar
  OK  Global results saved to: /content/fake-news-detector/experiments/results/short_text/all

[4/5] Training-only ranking...
  OK  Saved to: /content/fak